In [146]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [128]:
df = pd.read_csv("data/gnss_dataset_labeled.csv")

print(df.head())
print(df.shape)

   block_idx  time_s  best_prn  best_doppler  best_tau_chips  peak_snr  \
0         95   0.095        16          8000          701.48  5.275831   
1         15   0.015        23          -500          137.84  4.981328   
2         30   0.030         6         -9000          145.92  4.906167   
3         58   0.058        12          9000          536.56  4.903574   
4         85   0.085        11         -1500          404.36  5.001512   

     peak_raw  in_phase_lag  correlation_width  label label_name  
0  946.122544        457.72              13.52      0  authentic  
1  884.063116        339.12              14.52      0  authentic  
2  866.296813       -121.68              15.24      0  authentic  
3  788.611243        432.48              15.84      1    spoofed  
4  815.728799        285.48              13.32      1    spoofed  
(202, 11)


In [129]:
X = df.drop(columns=["label", "label_name"]).values
y = df["label"].values

In [130]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [131]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [132]:
class GNSSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [133]:
train_loader = DataLoader(GNSSDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(GNSSDataset(X_test, y_test), batch_size=32)

In [118]:
class CNN_LSTM(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.conv1 = nn.Conv1d(1, 64, 3)
        self.pool = nn.MaxPool1d(2)

        self.lstm = nn.LSTM(64, 64, batch_first=True)

        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = x.unsqueeze(1)

        x = torch.relu(self.conv1(x))
        x = self.pool(x)

        x = x.permute(0, 2, 1)

        _, (hn, _) = self.lstm(x)

        x = torch.relu(self.fc1(hn[-1]))
        x = self.fc2(x)

        return x

In [139]:
class CNN_Conv(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.conv1 = nn.Conv1d(1, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)

        self.conv2 = nn.Conv1d(64, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)

        self.pool = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.3)

        with torch.no_grad():
            self.eval()
            dummy = torch.zeros(1, 1, input_size)
            dummy = self.pool(self.bn1(torch.relu(self.conv1(dummy))))
            dummy = self.pool(self.bn2(torch.relu(self.conv2(dummy))))
            self.flatten_size = dummy.view(1, -1).shape[1]

            self.train()

        self.fc1 = nn.Linear(self.flatten_size, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = x.unsqueeze(1)

        x = self.pool(self.bn1(torch.relu(self.conv1(x))))
        x = self.pool(self.bn2(torch.relu(self.conv2(x))))

        x = x.view(x.size(0), -1)

        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [143]:
def train_model(model, train_loader, test_loader, epochs=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # TRAIN
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).float()

            optimizer.zero_grad()

            outputs = model(X_batch).view(-1)   
            loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    # TEST
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch).view(-1)
            preds = (torch.sigmoid(outputs) > 0.5).float()

            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    print("Test Accuracy:", correct / total)

In [144]:
print("=== CNN + LSTM ===")
model1 = CNN_LSTM(input_size=X_train.shape[1])
train_model(model1, train_loader, test_loader)

=== CNN + LSTM ===
Epoch 1, Loss: 4.1162
Epoch 2, Loss: 4.0934
Epoch 3, Loss: 4.1095
Epoch 4, Loss: 3.9487
Epoch 5, Loss: 3.9393
Epoch 6, Loss: 3.9079
Epoch 7, Loss: 3.7417
Epoch 8, Loss: 3.4403
Epoch 9, Loss: 3.3968
Epoch 10, Loss: 3.0021
Epoch 11, Loss: 2.9243
Epoch 12, Loss: 2.7065
Epoch 13, Loss: 2.4327
Epoch 14, Loss: 2.4085
Epoch 15, Loss: 2.3732
Epoch 16, Loss: 2.0785
Epoch 17, Loss: 2.0437
Epoch 18, Loss: 2.0391
Epoch 19, Loss: 2.2999
Epoch 20, Loss: 1.9557
Epoch 21, Loss: 1.7408
Epoch 22, Loss: 1.9652
Epoch 23, Loss: 1.6952
Epoch 24, Loss: 2.5446
Epoch 25, Loss: 1.8451
Epoch 26, Loss: 1.7338
Epoch 27, Loss: 1.6077
Epoch 28, Loss: 1.7275
Epoch 29, Loss: 1.6276
Epoch 30, Loss: 1.5261
Epoch 31, Loss: 1.4480
Epoch 32, Loss: 3.0726
Epoch 33, Loss: 1.7730
Epoch 34, Loss: 1.3513
Epoch 35, Loss: 1.3113
Epoch 36, Loss: 1.1888
Epoch 37, Loss: 1.2408
Epoch 38, Loss: 1.0096
Epoch 39, Loss: 0.8759
Epoch 40, Loss: 0.8679
Epoch 41, Loss: 0.7802
Epoch 42, Loss: 1.1594
Epoch 43, Loss: 0.8872
E

In [145]:
print("\n=== CNN + Conv1D ===")
model2 = CNN_Conv(input_size=X_train.shape[1])
train_model(model2, train_loader, test_loader)


=== CNN + Conv1D ===
Epoch 1, Loss: 4.2027
Epoch 2, Loss: 3.0424
Epoch 3, Loss: 2.6115
Epoch 4, Loss: 2.0643
Epoch 5, Loss: 2.4338
Epoch 6, Loss: 1.4648
Epoch 7, Loss: 1.2631
Epoch 8, Loss: 0.8110
Epoch 9, Loss: 0.6591
Epoch 10, Loss: 0.5170
Epoch 11, Loss: 0.3331
Epoch 12, Loss: 0.4776
Epoch 13, Loss: 0.2715
Epoch 14, Loss: 0.8038
Epoch 15, Loss: 0.4620
Epoch 16, Loss: 0.5193
Epoch 17, Loss: 0.4599
Epoch 18, Loss: 0.2355
Epoch 19, Loss: 1.0845
Epoch 20, Loss: 0.1672
Epoch 21, Loss: 0.4663
Epoch 22, Loss: 0.4781
Epoch 23, Loss: 0.1931
Epoch 24, Loss: 0.4377
Epoch 25, Loss: 0.0932
Epoch 26, Loss: 0.1746
Epoch 27, Loss: 0.2873
Epoch 28, Loss: 1.9046
Epoch 29, Loss: 0.6023
Epoch 30, Loss: 0.2087
Epoch 31, Loss: 0.2920
Epoch 32, Loss: 0.1860
Epoch 33, Loss: 0.1706
Epoch 34, Loss: 0.1306
Epoch 35, Loss: 0.1552
Epoch 36, Loss: 0.0786
Epoch 37, Loss: 0.0867
Epoch 38, Loss: 0.1362
Epoch 39, Loss: 3.3316
Epoch 40, Loss: 0.0940
Epoch 41, Loss: 0.2827
Epoch 42, Loss: 0.1423
Epoch 43, Loss: 2.138

In [148]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("\n=== RANDOM FOREST ===")
print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_prob_rf))


=== RANDOM FOREST ===
              precision    recall  f1-score   support

           0       0.86      1.00      0.93        19
           1       1.00      0.86      0.93        22

    accuracy                           0.93        41
   macro avg       0.93      0.93      0.93        41
weighted avg       0.94      0.93      0.93        41

Confusion Matrix:
 [[19  0]
 [ 3 19]]
ROC AUC: 0.992822966507177
